In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import sqlite3
import matplotlib as mpl
import matplotlib.pyplot as plt
import keras_tuner as kt

import copy
import random
import math

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder

from datetime import datetime

In [ ]:
dbConnectionString = "sqlite:///baseball_info.db"
engine = create_engine(dbConnectionString)

In [ ]:
hr%
fb%
gb%
hr/fb
hardHit%
barrel%
babip
swinggingg strike%
chasing strike %
foul strike%
fastball velo

In [ ]:
timeSeriesPitchingQuery = "SELECT SeasonStatsPitching.playerId,season,teams,Bios.dob,ipOuts,battersFaced,hits,singles,doubles,triples,homeRuns,runs,earnedRuns,walks,strikeOuts,hitByPitch,wildPitches,balks,stolenBases,caughtStealing,passedBalls,qualityStarts,averageFastballVelocity,zonePercentage,chasePercentage,swingingStrikePercentage,hardHitPercentage,barrelPercentage,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,era,whip,ksPerNine FROM SeasonStatsPitching INNER JOIN Bios on Bios.playerId = SeasonStatsPitching.playerId WHERE season BETWEEN 2015 and 2025 and season != 2020 and SeasonStatsPitching.gamesStarted > 0 and saves = 0 and holds = 0 ORDER BY SeasonStatsPitching.playerId,season"

df = pd.read_sql(timeSeriesPitchingQuery, engine)

In [ ]:
df.head()

In [ ]:
def calculate_age_from_timestamps(birth_timestamp, current_timestamp):
    birth_date   = datetime.fromtimestamp(birth_timestamp)
    current_date = datetime.fromtimestamp(current_timestamp)

    age = current_date.year - birth_date.year

    # Adjust age if the birthday hasn't occurred yet in the current year
    if (current_date.month, current_date.day) < (birth_date.month, birth_date.day):
        age -= 1
    
    return age

In [ ]:
twentyFifteenDateString     = "2015-04-05 00:00:00"
twentySixteenDateString     = "2016-04-03 00:00:00"
twentySeventeenDateString   = "2017-04-02 00:00:00"
twentyEighteenDateString    = "2018-03-29 00:00:00"
twentyNineteenDateString    = "2019-03-20 00:00:00"
twentyTwentyOneDateString   = "2021-04-01 00:00:00"
twentyTwentyTwoDateString   = "2022-04-07 00:00:00"
twentyTwentyThreeDateString = "2023-03-30 00:00:00"
twentyTwentyFourDateString  = "2024-03-20 00:00:00"
twentyTwentyFiveDateString  = "2025-03-18 00:00:00"

# Parse the string into a datetime object
twentyFifteenDateString     = datetime.strptime(twentyFifteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySixteenDateString     = datetime.strptime(twentySixteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySeventeenDateString   = datetime.strptime(twentySeventeenDateString   , "%Y-%m-%d %H:%M:%S")
twentyEighteenDateString    = datetime.strptime(twentyEighteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyNineteenDateString    = datetime.strptime(twentyNineteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyTwentyOneDateString   = datetime.strptime(twentyTwentyOneDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyTwoDateString   = datetime.strptime(twentyTwentyTwoDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyThreeDateString = datetime.strptime(twentyTwentyThreeDateString , "%Y-%m-%d %H:%M:%S")
twentyTwentyFourDateString  = datetime.strptime(twentyTwentyFourDateString  , "%Y-%m-%d %H:%M:%S")
twentyTwentyFiveDateString  = datetime.strptime(twentyTwentyFiveDateString  , "%Y-%m-%d %H:%M:%S")

# Convert to Unix timestamp
twentyFifteenTimeStamp     = int(twentyFifteenDateString     .timestamp())
twentySixteenTimeStamp     = int(twentySixteenDateString     .timestamp())
twentySeventeenTimeStamp   = int(twentySeventeenDateString   .timestamp())
twentyEighteenTimeStamp    = int(twentyEighteenDateString    .timestamp())
twentyNineteenTimeStamp    = int(twentyNineteenDateString    .timestamp())
twentyTwentyOneTimeStamp   = int(twentyTwentyOneDateString   .timestamp())
twentyTwentyTwoTimeStamp   = int(twentyTwentyTwoDateString   .timestamp())
twentyTwentyThreeTimeStamp = int(twentyTwentyThreeDateString .timestamp())
twentyTwentyFourTimeStamp  = int(twentyTwentyFourDateString  .timestamp())
twentyTwentyFiveTimeStamp  = int(twentyTwentyFiveDateString  .timestamp())

seasonToStartTimeStamp = {}

seasonToStartTimeStamp[2015] = twentyFifteenTimeStamp   
seasonToStartTimeStamp[2016] = twentySixteenTimeStamp   
seasonToStartTimeStamp[2017] = twentySeventeenTimeStamp   
seasonToStartTimeStamp[2018] = twentyEighteenTimeStamp   
seasonToStartTimeStamp[2019] = twentyNineteenTimeStamp   
seasonToStartTimeStamp[2021] = twentyTwentyOneTimeStamp  
seasonToStartTimeStamp[2022] = twentyTwentyTwoTimeStamp  
seasonToStartTimeStamp[2023] = twentyTwentyThreeTimeStamp
seasonToStartTimeStamp[2024] = twentyTwentyFourTimeStamp 
seasonToStartTimeStamp[2025] = twentyTwentyFiveTimeStamp

In [ ]:
#drop players that cannot create a valid window (ie have not played multiple seasons)
hasMultipleSeasons = df["playerId"].value_counts() > 1
df = df[df["playerId"].isin(hasMultipleSeasons[hasMultipleSeasons].index)]

#drop players with no plater appearances
df.drop(df[df['battersFaced'] == 0].index, inplace=True)

#get player ages
df['dob'] = df.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

In [ ]:
len(df)

In [ ]:
df['teams'] = df['teams'].str.split(',').str[0]

In [ ]:
df['teams']

In [ ]:
nonNullRowsVelocity = df[df['averageFastballVelocity'].notna()]

In [ ]:
meanVelocity = nonNullRowsVelocity['averageFastballVelocity'].mean()

meanVelocity

In [ ]:
df['averageFastballVelocity'] = df['averageFastballVelocity'].fillna(meanVelocity)

In [ ]:
nullRowsVelocity = df[df['averageFastballVelocity'].isna()]

nullRowsVelocity

In [ ]:
nonInfiniteRows = df[(df['era'] != 'Inf') & (df['whip'] != 'Inf')]

In [ ]:
maxEra  = nonInfiniteRows['era' ].max()
maxWhip = nonInfiniteRows['whip'].max()

In [ ]:
df['era'] = df['era'].replace(['Inf'], maxEra)
df['whip'] = df['whip'].replace(['Inf'], maxWhip)

In [ ]:
infiniteRows = df[df['playerId'] == '593833']

In [ ]:
len(df)

In [ ]:
featureNum = len(df.columns)

featureNum

In [ ]:
def getWindowedFeaturesAndLabels(frame, maxWindowSize, numberOfFeatures):
    frameArray = frame.values.tolist()

    inputs = []
    labels = []
    
    r = 0

    curWindow = []

    playerId = frameArray[0][0]
    
    while r < len(frameArray):
        curPlayerId = frameArray[r][0]

        if curPlayerId != playerId:
            window = []
            
            windowR = 0

            while windowR <= len(curWindow):
                if len(window) == maxWindowSize or windowR == len(curWindow):
                    label = window.pop()

                    labelTest = copy.deepcopy(label)

                    relevantLabels = labelTest[21:22] + labelTest[32:] 

                    if len(window) < maxWindowSize - 1:
                        diff = maxWindowSize - 1 - len(window)

                        while diff > 0:
                            padding = [-10000.0] * numberOfFeatures
                            window.append(padding)

                            diff -= 1

                    windowArray = [inner_list[2:] for inner_list in window]

                    inputs.append(copy.deepcopy(windowArray))
                    labels.append(copy.deepcopy(relevantLabels))

                    window.append(label)
                    window.pop(0)

                if windowR < len(curWindow):
                    window.append(curWindow[windowR])
                
                windowR += 1

            curWindow = []
            playerId  = curPlayerId

        curWindow.append(frameArray[r])

        r += 1

    return inputs,labels

In [ ]:
fullFeatures,fullLabels = getWindowedFeaturesAndLabels(df, 4, featureNum)

In [ ]:
for i in range(len(fullFeatures)):
    feature = fullFeatures[i]
    
    if len(feature) != 3:
        print(f"feature: {i}")

        break

    for j in range(len(feature)):
        window = feature[j]

        if len(window) != featureNum - 2:
            print(f"feature: {i}, window: {j}")
            i = len(fullFeatures) + 1
            
            break

In [ ]:
fullFeaturesArray = np.array(fullFeatures)
fullLabelsArray   = np.array(fullLabels)

In [ ]:
fullFeaturesArray.shape

In [ ]:
teamCategories = df['teams'].unique()

In [ ]:
teamCategories

In [ ]:
teamColumn = fullFeaturesArray[:,:,0].reshape(-1, 1)

teamColumn[2]

In [ ]:
oneHotEncoder = OneHotEncoder(sparse_output=False, categories=[teamCategories], handle_unknown="ignore")

In [ ]:
oneHotFlat = oneHotEncoder.fit_transform(teamColumn)

In [ ]:
oneHotFlat[1]

In [ ]:
oneHot3d = oneHotFlat.reshape(fullFeaturesArray.shape[0], fullFeaturesArray.shape[1], len(teamCategories))

In [ ]:
oneHot3d[0]

In [ ]:
fullFeaturesOneHot = np.concatenate((oneHot3d, fullFeaturesArray[:,:,1:]), axis=2)

In [ ]:
fullFeaturesArrayOneHotFloat = np.array(fullFeaturesOneHot, dtype=np.float32)

In [ ]:
fullFeaturesArrayOneHotFloat[0]

In [ ]:
qualityStartsBin = np.linspace(min(fullLabelsArray[:,0]), max(fullLabelsArray[:,0]), num=4)
eraBin           = np.linspace(min(fullLabelsArray[:,1]), max(fullLabelsArray[:,1]), num=4)
whipBin          = np.linspace(min(fullLabelsArray[:,2]), max(fullLabelsArray[:,2]), num=4)
ksPerNineBin     = np.linspace(min(fullLabelsArray[:,3]), max(fullLabelsArray[:,3]), num=4)

In [ ]:
qualityStartsBinned = np.digitize(fullLabelsArray[:,0], qualityStartsBin, right=True)
eraBinned           = np.digitize(fullLabelsArray[:,1], eraBin, right=True)
whipBinned          = np.digitize(fullLabelsArray[:,2], whipBin, right=True)
ksPerNineBinned     = np.digitize(fullLabelsArray[:,3], ksPerNineBin, right=True)

In [ ]:
fullLabelsArrayBinned = np.column_stack((qualityStartsBinned, eraBinned, whipBinned, ksPerNineBinned))

In [ ]:
binCount = {}

for label in fullLabelsArrayBinned:
    labelString = str(label[0]) + str(label[1]) + str(label[2]) + str(label[3])

    if labelString in binCount:
        binCount[labelString] = binCount[labelString] + 1
    else:
        binCount[labelString] = 1

In [ ]:
sortedBinCount = dict(sorted(binCount.items(), key=lambda item: item[1], reverse=True))

len(sortedBinCount)

In [ ]:
sortedBinCount

In [ ]:
labels = list(sortedBinCount.keys())
count  = list(sortedBinCount.values())

plt.figure(figsize=(16, 12))
bar = plt.bar(labels, count)

_ = plt.xticks(labels, rotation="vertical")
_ = plt.bar_label(bar, count)

In [ ]:
binIndexes = []

for label in labels:
    labelNumber = int(label)
    
    labelArray = []
    
    while labelNumber > 0:
        digit = labelNumber % 10
        
        labelArray.insert(0, digit)
        
        labelNumber = int(labelNumber / 10)
        
    while len(labelArray) < 4:
        labelArray.insert(0, 0)
        
    binIndexes.append(labelArray)

In [ ]:
binIndexes

In [ ]:
index = 0

for i in range(len(fullFeaturesArray[0])):
    for j in range(len(fullFeaturesArray[0][i])):
        print(f"{index}, {fullFeaturesArray[0][i][j]}")

        index += 1

In [ ]:
def addNoiseToPoints(point, label):
    noisyPoint = np.copy(point)
    
    mean               = 0
    stdDevWholeNumbers = 1
    stdDevPercentages  = 0.01
    
    wholeNumbersMask = np.zeros_like(noisyPoint, dtype=bool)
    
    allZerosForSecondWindow = (noisyPoint[33:66] == -10000.0).all()
    allZerosForThirdWindow  = (noisyPoint[66:  ] == -10000.0).all()
    
    wholeNumbersMask[1:20  ] = True
    
    if allZerosForSecondWindow != True:
        wholeNumbersMask[34:53] = True
        
    if allZerosForThirdWindow != True:
        wholeNumbersMask[67:86] = True
    
    percentageNumbersMask = np.zeros_like(noisyPoint, dtype=bool)
    
    percentageNumbersMask[20:33] = True
    
    if allZerosForSecondWindow != True:
        percentageNumbersMask[53:66] = True
        
    if allZerosForThirdWindow != True:
        percentageNumbersMask[86:  ] = True
    
    wholeNumbersToChange = np.sum(wholeNumbersMask)
    percentagesToChange  = np.sum(percentageNumbersMask)
    
    wholeNumberNoise      = np.random.normal(mean, stdDevWholeNumbers, size=wholeNumbersToChange)
    percentageNumberNoise = np.random.normal(mean, stdDevPercentages, size=percentagesToChange)
    
    noisyPoint[wholeNumbersMask     ] = np.maximum(0, noisyPoint[wholeNumbersMask     ] + wholeNumberNoise)
    noisyPoint[percentageNumbersMask] = np.maximum(0, noisyPoint[percentageNumbersMask] + percentageNumberNoise)
    
    for i in range(1, len(noisyPoint)):
        if wholeNumbersMask[i] == True:
            noisyPoint[i] = math.ceil(noisyPoint[i])
            
    startAge = noisyPoint[0]
    
    noisyPoint[34] = startAge + 1
    noisyPoint[67] = startAge + 2
            
    noisyLabel = np.copy(label)
            
    noisyPointWindowed = noisyPoint.reshape(3, 32)
    
    return noisyPointWindowed, noisyLabel

In [ ]:
numClusters = 4

interpolatedFeatures = []
interpolatedLabels   = []

for i in range(len(binIndexes)):
    binCondition = (fullLabelsArrayBinned[:, 0] == binIndexes[i][0]) & (fullLabelsArrayBinned[:, 1] == binIndexes[i][1]) & (fullLabelsArrayBinned[:, 2] == binIndexes[i][2]) & (fullLabelsArrayBinned[:, 3] == binIndexes[i][3])
    
    matchingLabelIndices = np.where(binCondition)[0]
    
    matchingFeatures = fullFeaturesArray[matchingLabelIndices]
    matchingLabels   = fullLabelsArray  [matchingLabelIndices]

    #if matching features is less than number of clusters, we either need to reduce the size of the clusters
    #or if its size 1, add gaussian noise n times
    
    matchingFeatureLen   = matchingFeatures.shape[0]
    matchingFeatures2d   = matchingFeatures.reshape(matchingFeatureLen, -1)
    
    actualNumClusters = numClusters
    
    if len(matchingFeatures2d) < 4:
        actualNumClusters = 1
        
    neededPoints = 336 - len(matchingFeatures2d)
    
    if neededPoints == 0:
        continue
    
    #if a cluster has length 1, we should exclude it and only use those which we can interpolate new points from
    kmeans = KMeans(n_clusters=actualNumClusters, random_state=0)

    matchingFeaturesWithoutTeam = [np.concatenate((row[1:33], row[34:66], row[67:])) for row in matchingFeatures2d]

    kmeans.fit(matchingFeaturesWithoutTeam)
    
    validClusterIndices = []
    
    for j in range(numClusters):
        clusterIndices = np.where(kmeans.labels_ == j)[0]
        numFeaturesInCluster = len(clusterIndices)
        
        #print(numFeaturesInCluster)
        
        if numFeaturesInCluster > 1:
            validClusterIndices.append(j)
            
    numValidClusters = len(validClusterIndices)
    
    if numValidClusters == 0:
        #print(f"smote naive: {len(matchingFeatures2d)}, {neededPoints}")
        
        while neededPoints > 0:
            noisyPoint, noisyLabel = addNoiseToPoints(matchingFeatures2d[0], matchingLabels[0])
            
            interpolatedFeatures.append(noisyPoint.tolist())
            interpolatedLabels  .append(noisyLabel.tolist())
            
            neededPoints -=1
            
        #print(len(interpolatedFeatures))
        #print(len(interpolatedLabels))
             
    else:
        neededPointsPerCluster = math.floor(neededPoints / numValidClusters)
        
        #print(f"smote unaive: {len(matchingFeatures2d)}, {neededPoints}, {neededPointsPerCluster}")
        
        newPointsInBin = 0
        
        for j in range(numValidClusters):
            clusterIndices            =  np.where(kmeans.labels_ == validClusterIndices[j])[0]
            matchingFeatures2dCluster = matchingFeatures2d[clusterIndices]
            matchingLabelsCluster     = matchingLabels    [clusterIndices]
            
            clusterLength = len(matchingFeatures2dCluster)
            pairsInCluster = (clusterLength * (clusterLength - 1))/2
            
            #print(f"cluster length: {clusterLength}")

            clusterRun = math.ceil(neededPointsPerCluster / pairsInCluster)
            
            for k in range(len(matchingFeatures2dCluster)):
                for l in range(k+1, len(matchingFeatures2dCluster)):
                    firstFeature = matchingFeatures2dCluster[k]
                    firstLabel   = matchingLabelsCluster    [k]

                    secondFeature = matchingFeatures2dCluster[l]
                    secondLabel   = matchingLabelsCluster    [l]
                    
                    pointsNeeded = clusterRun
                    
                    stepSize = 1.0 / (pointsNeeded + 1.0)
                    
                    interpolatedFeature = []
                    
                    #also need to check if we've already gotten enough points for this cluster
                    #if either point has masked values, prefer gaussian noise of the one without over interpolating
                    #if both points have masked values, fill in all zeros
                    while pointsNeeded > 0 and newPointsInBin < neededPoints:
                        interpolatedFeatureWindow = []
                        
                        for m in range(len(firstFeature)):
                            if m == 33 or m == 66:
                                interpolatedFeature.append(interpolatedFeatureWindow)
                                
                                interpolatedFeatureWindow = []
                            
                            firstFeaturePoint  = firstFeature [m]
                            secondFeaturePoint = secondFeature[m]

                            if firstFeaturePoint == -10000.0 and secondFeaturePoint != -10000.0:
                                noisySecondFeature,noisySecondLabel = addNoiseToPoints(secondFeature, secondLabel)

                                if m == 0:
                                    interpolatedFeature.append(noisySecondFeature[0])
                                    interpolatedFeature.append(noisySecondFeature[1])
                                    interpolatedFeature.append(noisySecondFeature[2])
                                elif m == 33:
                                    interpolatedFeature.append(noisySecondFeature[1])
                                    interpolatedFeature.append(noisySecondFeature[2])
                                else:
                                    interpolatedFeature.append(noisySecondFeature[2])
                                    
                                break

                                #print(interpolatedFeature)

                                #raise ExceptionType("An optional error message")

                            elif secondFeaturePoint == -10000.0 and firstFeaturePoint != -10000.0:
                                noisyFirstFeature,noisyFirstLabel = addNoiseToPoints(firstFeature, firstLabel)

                                if m == 0:
                                    interpolatedFeature.append(noisyFirstFeature[0])
                                    interpolatedFeature.append(noisyFirstFeature[1])
                                    interpolatedFeature.append(noisyFirstFeature[2])
                                elif m == 33:
                                    interpolatedFeature.append(noisyFirstFeature[1])
                                    interpolatedFeature.append(noisyFirstFeature[2])
                                else:
                                    interpolatedFeature.append(noisyFirstFeature[2])
                                    
                                break
                                
                                #print(noisyFirstFeature[0])
                                # print(f"{m}, {firstFeaturePoint}, {secondFeaturePoint}")
                                #print(interpolatedFeature)

                                #raise ExceptionType("An optional error message")
                            
                            elif m == 0 or m == 33 or m == 66:
                                randomTeamChoice = random.choice([True, False])

                                if randomTeamChoice:
                                    interpolatedFeatureWindow.append(firstFeaturePoint)
                                else:
                                    interpolatedFeatureWindow.append(secondFeaturePoint)

                            else:
                                slope = max(secondFeaturePoint, firstFeaturePoint) - min(secondFeaturePoint, firstFeaturePoint)
                                
                                interpolatedPoint = min(secondFeaturePoint, firstFeaturePoint) + slope * pointsNeeded * stepSize
                                
                                if m < 20 or (m >= 34 and m < 53) or (m >= 67 and m < 86):
                                    interpolatedPoint = math.floor(interpolatedPoint)
                                    
                                interpolatedFeatureWindow.append(interpolatedPoint)
                                
                                #print(f"{i}, {firstFeaturePoint}, {secondFeaturePoint}, {slope}, {pointsNeeded}, {stepSize}, {interpolatedPoint}")

                        if len(interpolatedFeature) < 3:
                            interpolatedFeature.append(interpolatedFeatureWindow)

                        startAge = interpolatedFeature[0][0]

                        if interpolatedFeature[1][0] != -10000.0:
                            interpolatedFeature[1][0] = startAge + 1
                        if interpolatedFeature[2][0] != -10000.0:
                            interpolatedFeature[2][0] = startAge + 2
    
                        interpolatedLabel = []
                        
                        for m in range(len(firstLabel)):
                            firstLabelPoint  = firstLabel [m]
                            secondLabelPoint = secondLabel[m]
                            
                            slope = max(secondLabelPoint, firstLabelPoint) - min(secondLabelPoint, firstLabelPoint)
                            
                            interpolatedPoint = min(secondLabelPoint, firstLabelPoint) + slope * pointsNeeded * stepSize
                    
                            if m == 0:
                                interpolatedPoint = math.floor(interpolatedPoint)
                                
                            interpolatedLabel.append(interpolatedPoint)
                            
                            #print(f"{i}, {firstLabelPoint}, {secondLabelPoint}, {slope}, {interpolatedPoint}")
                            
                        pointsNeeded -= 1
                        newPointsInBin += 1

                        interpolatedFeatures.append(interpolatedFeature)
                        interpolatedLabels  .append(interpolatedLabel  )
                        
                        interpolatedFeature = []
                        
        #print(len(interpolatedFeatures))
        #print(len(interpolatedLabels))

In [ ]:
interpolatedLabels[1100]

In [ ]:
interpolatedFeaturesArray = np.array(interpolatedFeatures)
interpolatedLabelsArray   = np.array(interpolatedLabels)

In [ ]:
concatenatedFeaturesArray = np.concatenate((fullFeaturesArray, interpolatedFeaturesArray), axis=0)
concatenatedLabelsArray   = np.concatenate((fullLabelsArray  , interpolatedLabelsArray  ), axis=0)

In [ ]:
concatenatedQualityStartsBins = np.linspace(min(concatenatedLabelsArray[:,0]), max(concatenatedLabelsArray[:,0]), num=4)
concatenatedEraBins           = np.linspace(min(concatenatedLabelsArray[:,1]), max(concatenatedLabelsArray[:,1]), num=4)
concatenatedWhipBins          = np.linspace(min(concatenatedLabelsArray[:,2]), max(concatenatedLabelsArray[:,2]), num=4)
concatenatedKsPerNineBins     = np.linspace(min(concatenatedLabelsArray[:,3]), max(concatenatedLabelsArray[:,3]), num=4)

concatenatedQualityStartsBinned = np.digitize(concatenatedLabelsArray[:,0], concatenatedQualityStartsBins, right=True)
concatenatedEraBinned           = np.digitize(concatenatedLabelsArray[:,1], concatenatedEraBins, right=True)
concatenatedWhipBinned          = np.digitize(concatenatedLabelsArray[:,2], concatenatedWhipBins, right=True)
concatenatedKsPerNineBinned     = np.digitize(concatenatedLabelsArray[:,3], concatenatedKsPerNineBins, right=True)

fullConcatenatedLabelsArrayBinned = np.column_stack((concatenatedQualityStartsBinned, concatenatedEraBinned, concatenatedWhipBinned, concatenatedKsPerNineBinned))

In [ ]:
concatedBinCount = {}

for label in fullConcatenatedLabelsArrayBinned:
    labelString = str(label[0]) + str(label[1]) + str(label[2]) + str(label[3])

    if labelString in concatedBinCount:
        concatedBinCount[labelString] = concatedBinCount[labelString] + 1
    else:
        concatedBinCount[labelString] = 1

In [ ]:
concatenatedLabels = list(concatedBinCount.keys())
concatenatedCount  = list(concatedBinCount.values())

plt.figure(figsize=(16, 12))
bar = plt.bar(concatenatedLabels, concatenatedCount)

_ = plt.xticks(concatenatedLabels, rotation="vertical")
_ = plt.bar_label(bar, concatenatedCount)

In [ ]:
#NEED TO CHANGE IMPLEMEENTATION OR ORDERING OF MASKING TO AVOID NORMALIZING MASKED VALUES

featureMask = concatenatedFeaturesArray == -10000.0

maskedFeatureArray = np.ma.masked_array(concatenatedFeaturesArray, mask=featureMask)

featuresMean = maskedFeatureArray.mean(axis=(0,1), keepdims=True)
featuresStd  = maskedFeatureArray.std(axis=(0,1), keepdims=True)

print(featuresMean[0][0][31])

labelMeansList = [[featuresMean[0][0][18], featuresMean[0][0][29], featuresMean[0][0][30], featuresMean[0][0][31]]]
labelStdList   = [[featuresStd [0][0][18], featuresStd [0][0][29], featuresStd [0][0][30], featuresStd [0][0][31]]]  

labelsMean = np.array(labelMeansList)
labelsStd  = np.array(labelStdList)

normalizedFeaturesMasked = (maskedFeatureArray - featuresMean) / (featuresStd)
normalizedLabels         = (concatenatedLabelsArray - labelsMean) / (labelsStd)

normalizedFeatures = normalizedFeaturesMasked.data

In [ ]:
featuresStd

In [ ]:
normalizedLabels[2]

In [ ]:
trainFeatures,valTestFeatures,trainLabels,valTestLables = train_test_split(normalizedFeatures, normalizedLabels, stratify=fullConcatenatedLabelsArrayBinned, test_size=0.2)

valFeatures,testFeatures,valLabels,testLabels = train_test_split(valTestFeatures, valTestLables, test_size=0.5)

In [ ]:
trainFeatures.shape

In [ ]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization
def getModelForTuner(hp):
    hp_learning_rate     = hp.Float('learning_rate'    , min_value=1e-7, max_value=1e-3, sampling='log')
    hp_weight_decay      = hp.Float('weight_decay'     , min_value=1e-7, max_value=1e-3, sampling='log')   

    num_rnn_layers = hp.Int("num_rnn_layers", min_value=1, max_value=3)
    
    model = tf.keras.Sequential()
    model.add(layers.Masking(mask_value=-10000.0, input_shape=(3, 32)))

    for i in range(num_rnn_layers):
        hp_units             = hp.Int   (f'units_{i}', min_value=5, max_value=30, step=6)
        hp_dropout           = hp.Float(f'dropout_{i}'          , min_value=0.1, max_value=0.9, sampling='linear')     
        hp_recurrent_dropout = hp.Float(f'recurrent_dropout_{i}', min_value=0.1, max_value=0.9, sampling='linear')  
        hp_regularizer       = hp.Float(f'regularizer_{i}'      , min_value=1e-7, max_value=1e-3, sampling='log')  

        if i < num_rnn_layers - 1:
            model.add(layers.Bidirectional(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=True)))
        else:
            model.add(layers.Bidirectional(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=False)))
        
        model.add(layers.LayerNormalization())
        
        model.add(layers.Dense(units = 4))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate, weight_decay=hp_weight_decay), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
tuner = kt.Hyperband(getModelForTuner,
                     objective='val_loss',
                     max_epochs=100,
                     factor=3,
                     directory='starting_pitcher_tuner',
                     project_name='starting_pitcher_tuner_test_2')

In [ ]:
earlyStop = EarlyStopping(monitor='val_loss', mode='min', patience=10, restore_best_weights=True)

In [ ]:
tuner.search(trainFeatures, trainLabels, batch_size=64, epochs=100, validation_data=(valFeatures, valLabels), callbacks=[earlyStop])

In [ ]:
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
print(best_hps.get('num_rnn_layers'))
print(best_hps.get('learning_rate'))
print(best_hps.get('weight_decay'))
print(best_hps.get('units_0'))
print(best_hps.get('dropout_0'))
print(best_hps.get('recurrent_dropout_0'))
print(best_hps.get('regularizer_0'))

print("-----------------")

print(best_hps.get('units_1'))
print(best_hps.get('dropout_1'))
print(best_hps.get('recurrent_dropout_1'))
print(best_hps.get('regularizer_1'))

print("--------------------------")

print(best_hps.get('units_2'))
print(best_hps.get('dropout_2'))
print(best_hps.get('recurrent_dropout_2'))
print(best_hps.get('regularizer_2'))


In [ ]:
def get_bidirectional_model():
    model = tf.keras.Sequential([
        layers.Masking(mask_value=-10000.0, input_shape=(3, 32)),
        #normalizeInput,
        layers.Bidirectional(layers.LSTM(units=29, dropout=0.5465739351328269, recurrent_dropout=0.717150584044477, kernel_regularizer=regularizers.l2(4.6992071620981e-05), return_sequences=True)),
        layers.LayerNormalization(),
        layers.Bidirectional(layers.LSTM(units=23, dropout=0.11235996300102809, recurrent_dropout=0.5978356244592431, kernel_regularizer=regularizers.l2(0.0002692731431927674), return_sequences=False)),
        layers.LayerNormalization(),
        layers.Dense(units = 4)
    ])

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0007527457083097, weight_decay=3.1198275002615033e-07), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
curModel = get_bidirectional_model()

curModel.summary()

In [ ]:
curModel.fit(trainFeatures, trainLabels, batch_size=64, epochs=100, validation_data=(valFeatures, valLabels), callbacks=[earlyStop])

In [ ]:
baselineResults = curModel.evaluate(testFeatures, testLabels)

In [ ]:
accuracy: 0.8218 - loss: 0.1291 - r2_score: 0.9552

In [ ]:
#TestResults
playerFeaturesQuery = "SELECT SeasonStatsPitching.playerId,season,Bios.dob,ipOuts,battersFaced,hits,singles,doubles,triples,homeRuns,runs,earnedRuns,walks,strikeOuts,hitByPitch,wildPitches,balks,stolenBases,caughtStealing,passedBalls,qualityStarts,averageFastballVelocity,zonePercentage,chasePercentage,swingingStrikePercentage,hardHitPercentage,barrelPercentage,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,era,whip,ksPerNine FROM SeasonStatsPitching INNER JOIN Bios on Bios.playerId = SeasonStatsPitching.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsPitching.playerId =\"702056\" ORDER BY SeasonStatsPitching.playerId,season"

playerDf = pd.read_sql(playerFeaturesQuery, engine)

In [ ]:
playerDf['dob'] = playerDf.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

In [ ]:
playerDf.transpose()

In [ ]:
playerStats = []

playerFrameArray = playerDf.values.tolist()

for i in range(len(playerFrameArray)):
    playerStats.append(playerFrameArray[i][2:])

if len(playerStats) < 3:
    diff = 3 - len(playerStats)

    while diff > 0:
        playerStats.append([-10000.0] * 32)

        diff -= 1

In [ ]:
playerWindow = np.array([playerStats])

playerWindowMask = playerWindow == -10000.0

maskedPlayerWindow = np.ma.masked_array(playerWindow, mask=playerWindowMask)

playerWindowMaskNormalized = (maskedPlayerWindow - featuresMean) / (featuresStd)

playerWindowNormalized = playerWindowMaskNormalized.data

In [ ]:
playerWindowNormalized

In [ ]:
playerPrediction = curModel.predict(playerWindowNormalized)

In [ ]:
indexToMean = {}

indexToMean[0] = labelsMean[0][0]
indexToMean[1] = labelsMean[0][1]
indexToMean[2] = labelsMean[0][2]
indexToMean[3] = labelsMean[0][3]

indexToStd = {}

indexToStd[0] = labelsStd[0][0]
indexToStd[1] = labelsStd[0][1]
indexToStd[2] = labelsStd[0][2]
indexToStd[3] = labelsStd[0][3]

for i in range(len(playerPrediction[0])):
    prediction = playerPrediction[0][i]

    curMean = indexToMean[i]
    curStd  = indexToStd [i]

    #print(f"{curMean}, {curStd}")

    prediction = prediction * curStd + curMean

    #value * std of column + mean of column
    #prediction = math.exp(prediction) - 1

    if i == 0:
        print(math.ceil(prediction))
    else:
        print(round(prediction, 3))

In [ ]:
#curModel.save_weights('./pitcherModelCheckPoints/model_1-31-2026.weights.h5')

In [ ]:
#763
indexToMean = {}

indexToMean[0] = labelsMean[0][0]
indexToMean[1] = labelsMean[0][1]
indexToMean[2] = labelsMean[0][2]
indexToMean[3] = labelsMean[0][3]

indexToStd = {}

indexToStd[0] = labelsStd[0][0]
indexToStd[1] = labelsStd[0][1]
indexToStd[2] = labelsStd[0][2]
indexToStd[3] = labelsStd[0][3]

season = 2025

hitterIds = []

hitterIdQuery = f"SELECT DISTINCT(SeasonStatsPitching.playerId),firstName,lastName from SeasonStatsPitching INNER JOIN Bios ON Bios.playerId = SeasonStatsPitching.playerId WHERE SeasonStatsPitching.season={season} and SeasonStatsPitching.gamesStarted > 0 and SeasonStatsPitching.saves = 0 and SeasonStatsPitching.holds = 0"

connection = sqlite3.connect("C:/baseball_info.db")
cursor     = connection.cursor()

cursor.execute(hitterIdQuery)

rows = cursor.fetchall()

for row in rows:
    hitterIds.append((row[0],row[1],row[2]))

cursor    .close()
connection.close()

print(len(hitterIds))

In [ ]:
playerPredictions = []

for hitter in hitterIds:
    playerId = hitter[0]

    playerFeaturesQuery = f"SELECT SeasonStatsPitching.playerId,season,Bios.dob,ipOuts,battersFaced,hits,singles,doubles,triples,homeRuns,runs,earnedRuns,walks,strikeOuts,hitByPitch,wildPitches,balks,stolenBases,caughtStealing,passedBalls,qualityStarts,averageFastballVelocity,zonePercentage,chasePercentage,swingingStrikePercentage,hardHitPercentage,barrelPercentage,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,era,whip,ksPerNine FROM SeasonStatsPitching INNER JOIN Bios on Bios.playerId = SeasonStatsPitching.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsPitching.playerId =\"{playerId}\" ORDER BY SeasonStatsPitching.playerId,season"

    playerDf = pd.read_sql(playerFeaturesQuery, engine)

    playerDf['dob'] = playerDf.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

    playerStats = []

    playerFrameArray = playerDf.values.tolist()
    
    for i in range(len(playerFrameArray)):
        playerStats.append(playerFrameArray[i][2:])
    
    if len(playerStats) < 3:
        diff = 3 - len(playerStats)
    
        while diff > 0:
            playerStats.append([-10000.0] * 32)
    
            diff -= 1

    playerWindow = np.array([playerStats])

    playerWindowMask = playerWindow == -10000.0
    
    maskedPlayerWindow = np.ma.masked_array(playerWindow, mask=playerWindowMask)
    
    playerWindowMaskNormalized = (maskedPlayerWindow - featuresMean) / (featuresStd)
    
    playerWindowNormalized = playerWindowMaskNormalized.data

    playerPrediction = curModel.predict(playerWindowNormalized)

    normalizedPlayerPrediction = (playerId, hitter[1], hitter[2])

    for i in range(len(playerPrediction[0])):
        prediction = playerPrediction[0][i]
    
        curMean = indexToMean[i]
        curStd  = indexToStd [i]
    
        prediction = prediction * curStd + curMean
    
        if i == 0:
            prediction = max(0, math.ceil(prediction))
        else:
            prediction = max(0.0, round(prediction, 3))

        normalizedPlayerPrediction += (prediction,)

    playerPredictions.append(normalizedPlayerPrediction)

In [ ]:
c

In [ ]:
maxQualityStarts = 0.0
minQualityStarts = 10000.0

maxEra = 0.0
minEra = 10000.0

maxWhip = 0.0
minWhip = 10000.0

maxKsPerNine = 0.0
minKsPerNine = 10000.0

for player in playerPredictions:
    qualityStarts = player[3]
    era           = player[4]
    whip          = player[5]
    ksPerNine     = player[6]

    maxQualityStarts = max(maxQualityStarts, qualityStarts)
    minQualityStarts = min(minQualityStarts, qualityStarts)

    maxEra = max(maxEra, era)
    minEra = min(minEra, era)

    maxWhip = max(maxWhip, whip)
    minWhip = min(minWhip, whip)

    maxKsPerNine = max(maxKsPerNine, ksPerNine)
    minKsPerNine = min(minKsPerNine, ksPerNine)

    
print(f"{maxQualityStarts}, {minQualityStarts}")
print(f"{maxEra}, {minEra}")
print(f"{maxWhip}, {minWhip}")
print(f"{maxKsPerNine}, {minKsPerNine}")

In [ ]:
def normalize(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    normalizedValue = (num - minValue) / (maxValue - minValue)

    return normalizedValue

In [ ]:
def reverse_normalization(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    reverseNormalizedValue = 1 - ((num - minValue) / (maxValue - minValue))

    return reverseNormalizedValue

In [ ]:
for i in range(len(playerPredictions)):
    player = playerPredictions[i]
    
    qualityStarts = player[3]
    era           = player[4]
    whip          = player[5]
    ksPerNine     = player[6]

    normalizedQualityStarts = normalize            (qualityStarts, maxQualityStarts, minQualityStarts)
    normalizedEra           = reverse_normalization(era, maxEra, minEra)
    normalizedWhip          = reverse_normalization(whip, maxWhip, minWhip)
    normalizedKsPerNine     = normalize            (ksPerNine, maxKsPerNine, minKsPerNine)

    overallGrade = normalizedEra + normalizedWhip + normalizedKsPerNine

    player += (overallGrade,)

    playerPredictions[i] = player

In [ ]:
playerPredictions[0]

In [ ]:
gradeSortedPredictions = sorted(playerPredictions, key=lambda player: player[7], reverse=True)

In [ ]:
for i in range(50):
    print(gradeSortedPredictions[i])

In [ ]:
predictionHeaders = ["id", "firstName", "lastName", "qualityStarts", "era", "whip", "ksPerNine", "grade"]

filePath = "./starting_pitcher_predictions_2026_3.csv"

predictionsDf = pd.DataFrame(gradeSortedPredictions, columns=predictionHeaders)

predictionsDf.to_csv(filePath, index=False)

In [ ]:
reliefPitcherIds = []

reliefPitcherIdQuery = "SELECT DISTINCT(SeasonGradesReliefPitchers.playerId),firstName,lastName from SeasonGradesReliefPitchers INNER JOIN Bios on Bios.playerId = SeasonGradesReliefPitchers.playerId where SeasonGradesReliefPitchers.season=2025"

connection = sqlite3.connect("C:/baseball_info.db")
cursor     = connection.cursor()

cursor.execute(reliefPitcherIdQuery)

rows = cursor.fetchall()

for row in rows:
    reliefPitcherIds.append((row[0],row[1],row[2]))

cursor    .close()
connection.close()

print(len(reliefPitcherIds))

In [ ]:
reliefPitcherIds[0]

In [ ]:
reliefPitcherPredictions = []

for pitcher in reliefPitcherIds:
    playerId = pitcher[0]

    playerFeaturesQuery = f"SELECT SeasonStatsPitching.playerId,season,Bios.dob,ipOuts,battersFaced,hits,singles,doubles,triples,homeRuns,runs,earnedRuns,walks,strikeOuts,hitByPitch,wildPitches,balks,stolenBases,caughtStealing,passedBalls,averageFastballVelocity,zonePercentage,chasePercentage,swingingStrikePercentage,hardHitPercentage,barrelPercentage,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,saves,holds,qualityStarts,era,whip,ksPerNine FROM SeasonStatsPitching INNER JOIN Bios on Bios.playerId = SeasonStatsPitching.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsPitching.playerId =\"{playerId}\" ORDER BY SeasonStatsPitching.playerId,season"

    playerDf = pd.read_sql(playerFeaturesQuery, engine)

    playerDf['dob'] = playerDf.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

    playerDf['averageFastballVelocity'] = playerDf['averageFastballVelocity'].fillna(meanVelocity)
    
    playerStats = []

    playerFrameArray = playerDf.values.tolist()
    
    for i in range(len(playerFrameArray)):
        playerStats.append(playerFrameArray[i][2:])
    
    if len(playerStats) < 3:
        diff = 3 - len(playerStats)
    
        while diff > 0:
            playerStats.append([-10000.0] * 34)
    
            diff -= 1

    playerWindow = np.array([playerStats])

    playerWindowMask = playerWindow == -10000.0
    
    maskedPlayerWindow = np.ma.masked_array(playerWindow, mask=playerWindowMask)
    
    playerWindowMaskNormalized = (maskedPlayerWindow - featuresMean) / (featuresStd)
    
    playerWindowNormalized = playerWindowMaskNormalized.data

    playerPrediction = curModel.predict(playerWindowNormalized)

    normalizedPlayerPrediction = (playerId, pitcher[1], pitcher[2])

    for i in range(len(playerPrediction[0])):
        prediction = playerPrediction[0][i]
    
        curMean = indexToMean[i]
        curStd  = indexToStd [i]
    
        prediction = prediction * curStd + curMean
    
        if i < 3:
            prediction = max(0, math.ceil(prediction))
        else:
            prediction = max(0.0, round(prediction, 3))

        normalizedPlayerPrediction += (prediction,)

    reliefPitcherPredictions.append(normalizedPlayerPrediction)

In [ ]:
maxSavesAndHolds = 0.0
minSavesAndHolds = 10000.0

maxEra = 0.0
minEra = 10000.0

maxWhip = 0.0
minWhip = 10000.0

maxKsPerNine = 0.0
minKsPerNine = 10000.0

for player in reliefPitcherPredictions:
    saves         = player[3]
    holds         = player[4]
    era           = player[6]
    whip          = player[7]
    ksPerNine     = player[8]

    maxSavesAndHolds = max(maxSavesAndHolds, saves + holds)
    minSavesAndHolds = min(minSavesAndHolds, saves + holds)

    maxEra = max(maxEra, era)
    minEra = min(minEra, era)

    maxWhip = max(maxWhip, whip)
    minWhip = min(minWhip, whip)

    maxKsPerNine = max(maxKsPerNine, ksPerNine)
    minKsPerNine = min(minKsPerNine, ksPerNine)

    
print(f"{maxSavesAndHolds}, {minSavesAndHolds}")
print(f"{maxEra}, {minEra}")
print(f"{maxWhip}, {minWhip}")
print(f"{maxKsPerNine}, {minKsPerNine}")

In [ ]:
for i in range(len(reliefPitcherPredictions)):
    player = reliefPitcherPredictions[i]
    
    saves         = player[3]
    holds         = player[4]
    era           = player[6]
    whip          = player[7]
    ksPerNine     = player[8]

    normalizedSavesAndHolds = normalize            (saves + holds, maxSavesAndHolds, minSavesAndHolds)
    normalizedEra           = reverse_normalization(era, maxEra, minEra)
    normalizedWhip          = reverse_normalization(whip, maxWhip, minWhip)
    normalizedKsPerNine     = normalize            (ksPerNine, maxKsPerNine, minKsPerNine)

    overallGrade = normalizedSavesAndHolds + normalizedEra + normalizedWhip + normalizedKsPerNine

    player += (overallGrade,)

    reliefPitcherPredictions[i] = player

In [ ]:
gradeSortedReliefPredictions = sorted(reliefPitcherPredictions, key=lambda player: player[9], reverse=True)

In [ ]:
predictionHeaders = ["id", "firstName", "lastName", "saves", "holds", "qualityStarts", "era", "whip", "ksPerNine", "grade"]

filePath = "./relief_pitcher_predictions_2026.csv"

predictionsDf = pd.DataFrame(gradeSortedReliefPredictions, columns=predictionHeaders)

predictionsDf.to_csv(filePath, index=False)